In [ ]:
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.metrics import (
    matthews_corrcoef, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix
)
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
import shutil

import sys
sys.path.append('.')
from config import PROJECT_DIR, DATASET_ZIP, DATASET_DIR, MODELS_DIR, LABELS, MODELS, CONFIGS

In [ ]:
!unzip -q {DATASET_ZIP} -d {PROJECT_DIR}

## Reproducibility

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {DEVICE}')

In [ ]:
# N-CLAHE
def apply_nclahe(images_np, tile_size):
    images_log = np.log1p(images_np.astype(np.float32))
    images_log = ((images_log - images_log.min()) /
                  (images_log.max() - images_log.min()) * 255).astype(np.uint8)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(tile_size, tile_size))
    return clahe.apply(images_log)


class ChestXrayDataset(Dataset):
    def __init__(self, metadata_csv, split_csv, images_dir, resolution):
        self.df = pd.read_csv(metadata_csv)
        self.split_ids = pd.read_csv(split_csv)['image_id'].tolist()
        self.df = self.df[self.df['image_id'].isin(self.split_ids)]
        self.images_dir = images_dir
        self.resolution = resolution
        self.tile_size = 4 if resolution == 256 else 16
        self.image_ids = self.df['image_id'].unique().tolist()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img = cv2.imread(f'{self.images_dir}/{image_id}.png', cv2.IMREAD_GRAYSCALE)
        img = apply_nclahe(img, self.tile_size)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = Image.fromarray(img)
        img = transforms.ToTensor()(img)
        img = self.normalize(img)
        rows = self.df[self.df['image_id'] == image_id]
        label = torch.zeros(2)
        if 0 in rows['class_id'].values:
            label[0] = 1.0
        if 1 in rows['class_id'].values:
            label[1] = 1.0
        return img, label, image_id

In [ ]:
def build_model(architecture, resolution):
    if architecture == 'alexnet':
        model = models.alexnet(weights=None)
        model.classifier[6] = nn.Linear(4096, 2)
    elif architecture == 'densenet':
        model = models.densenet121(weights=None)
        model.classifier = nn.Linear(1024, 2)
    return model.to(DEVICE)

In [ ]:
# Clinical evaluation function for each model

def clinical_eval_model(model, test_loader):
    model.eval()
    all_logits = []
    all_labels = []

    with torch.no_grad():
        for imgs, labels, _ in test_loader:
            imgs = imgs.to(DEVICE)
            outputs = model(imgs)
            # Save logits (before sigmoid) for AUC
            all_logits.append(outputs.cpu().numpy())
            all_labels.append(labels.numpy())

    all_logits = np.concatenate(all_logits)     # (N, 2)
    all_labels = np.concatenate(all_labels)     # (N, 2)
    all_probs  = 1 / (1 + np.exp(-all_logits))  # sigmoid

    results = {}

    for i, name in enumerate(LABELS):
        y_true = all_labels[:, i]
        y_prob = all_probs[:, i]
    
        # Optimal threshold by MCC over ROC 
        fpr, tpr, thresholds = roc_curve(y_true, y_prob)
        mccs = []
        for t in thresholds:
            y_pred_t = (y_prob >= t).astype(int)
            mccs.append(matthews_corrcoef(y_true, y_pred_t))
        optimal_threshold = thresholds[np.argmax(mccs)]
        y_pred = (y_prob >= optimal_threshold).astype(int)

        # Clinical metrics
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
        mcc = matthews_corrcoef(y_true, y_pred)
        auc_roc = roc_auc_score(y_true, y_prob)
        pr_auc  = average_precision_score(y_true, y_prob)

        results[name] = {
            'optimal threshold': round(optimal_threshold, 4),
            'MCC':               round(mcc, 4),
            'AUC-ROC':           round(auc_roc, 4),
            'PR-AUC':            round(pr_auc, 4),
            'recall':            round(recall, 4),
            'specificity':       round(specificity, 4),
            'TP': int(tp), 'TN': int(tn), 'FP': int(fp), 'FN': int(fn)
        }

    return results

In [ ]:
all_results = {}
global_thresholds = {}

for cfg in CONFIGS:

    print(f"\n{'='*50}")
    print(f"Evaluating {cfg['name']}...")

    # Dataset and loader
    test_dataset = ChestXrayDataset(
        metadata_csv=f"{DATASET_DIR}/{cfg['metadata']}",
        split_csv=f"{DATASET_DIR}/split_test.csv",
        images_dir=f"{DATASET_DIR}/{cfg['images']}",
        resolution=cfg['res']
    )
    test_loader = DataLoader(test_dataset, batch_size=cfg['batch'], shuffle=False)
    print(f"Test set: {len(test_dataset)} images")

    # Load model
    model = build_model(cfg['arq'], cfg['res'])
    pth_path = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth_path, map_location=DEVICE))
    print(f"Checkpoint loaded: {pth_path}")

    # Evaluate
    results = clinical_eval_model(model, test_loader)
    global_thresholds[cfg['name']] = {
    'aneurysm': results['aneurysm']['optimal threshold'],
    'cardiomegaly': results['cardiomegaly']['optimal threshold']
    }
    all_results[cfg['name']] = results

    # Print results
    for label, metrics in results.items():
        print(f"\n  {label}:")
        for k, v in metrics.items():
            print(f"    {k}: {v}")

print("\n✓ Evaluation completed.")

In [ ]:
# Comparative summary table
rows = []
for model, labels in all_results.items():
    for label, metrics in labels.items():
        rows.append({
            'Model': model,
            'Label': label,
            'MCC': metrics['MCC'],
            'AUC-ROC': metrics['AUC-ROC'],
            'PR-AUC': metrics['PR-AUC'],
            'Recall': metrics['recall'],
            'Specificity': metrics['specificity'],
            'Threshold': metrics['optimal threshold']
        })

df_results = pd.DataFrame(rows)
print(df_results.to_string(index=False))

# Save to PROJECT_DIR
df_results.to_csv(f'{PROJECT_DIR}/results_clinical_evaluation.csv', index=False)
print(f'\n✓ Saved: {PROJECT_DIR}/results_clinical_evaluation.csv')

In [ ]:
# Save predictions by image
registers = []

for cfg in CONFIGS:
    test_dataset = ChestXrayDataset(
        metadata_csv=f"{DATASET_DIR}/{cfg['metadata']}",
        split_csv=f"{DATASET_DIR}/split_test.csv",
        images_dir=f"{DATASET_DIR}/{cfg['images']}",
        resolution=cfg['res']
    )
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

    model = build_model(cfg['arq'], cfg['res'])
    pth_path = f"{MODELS_DIR}/{cfg['arq']}_{cfg['res']}_best.pth"
    model.load_state_dict(torch.load(pth_path, map_location=DEVICE))
    model.eval()

    u = global_thresholds[cfg['name']]

    with torch.no_grad():
        for imgs, labels, image_ids in test_loader:
            imgs = imgs.to(DEVICE)
            output = model(imgs)
            probs = torch.sigmoid(output).cpu().numpy()[0]
            label = labels.numpy()[0]
            image_id = image_ids[0]

            registers.append({
                'model': cfg['name'],
                'image_id': image_id,
                'prob_aneurysm': round(float(probs[0]), 4),
                'prob_cardiomegaly': round(float(probs[1]), 4),
                'pred_aneurysm': int(probs[0] >= u['aneurysm']),
                'pred_cardiomegaly': int(probs[1] >= u['cardiomegaly']),
                'label_aneurysm': int(label[0]),
                'label_cardiomegaly': int(label[1]),
            })

df_preds = pd.DataFrame(registers)
df_preds.to_csv(f"{PROJECT_DIR}/predictions_by_image.csv", index=False)
print(f'✓ Saved: {len(df_preds)} rows to predictions_by_image.csv')

## Approach to clinical valid threshold

In [ ]:
print("Minimum lower bound to have FN=0 in aortic enlargement by model:\n")

for model in ['AN_256', 'DN_256', 'AN_1024', 'DN_1024']:
    df_m = df_preds[df_preds['model'] == model].copy()

    # Only images with real aortic enlargement
    df_aneu = df_m[df_m['label_aneurysm'] == 1]

    # Minimum threshold = minimum probability among true positives
    threshold_fn0 = df_aneu['prob_aneurysm'].min()

    # With that threshold, calculate FP and specificity
    df_m['pred_new'] = (df_m['prob_aneurysm'] >= threshold_fn0).astype(int)

    TN = ((df_m['pred_new'] == 0) & (df_m['label_aneurysm'] == 0)).sum()
    FP = ((df_m['pred_new'] == 1) & (df_m['label_aneurysm'] == 0)).sum()
    TP = ((df_m['pred_new'] == 1) & (df_m['label_aneurysm'] == 1)).sum()
    FN = ((df_m['pred_new'] == 0) & (df_m['label_aneurysm'] == 1)).sum()

    specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
    recall  = TP / (TP + FN) if (TP + FN) > 0 else 0

    print(f'{model}:')
    print(f'  Minimum threshold FN=0: {threshold_fn0:.4f}')
    print(f'  Recall: {recall:.4f} | Specificity: {specificity:.4f}')
    print(f'  TP: {TP} | FP: {FP} | TN: {TN} | FN: {FN}\n')

## Restore

In [ ]:
print(f'Removing {DATASET_DIR}...')
shutil.rmtree(DATASET_DIR)